<a href="https://colab.research.google.com/github/aayushrouniyarcs24-cmd/AI-Lab-1BF24CS010/blob/main/1BF24CS010_Week4_HillClimbing_N_Queens.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
import random

def get_heuristics_coords(coords):
    """Calculate the number of mutually attacking pairs of queens using direct 2D coordinates (col, row)."""
    attacks = 0
    n = len(coords)
    for i in range(n):
        c1, r1 = coords[i]
        for j in range(i + 1, n):
            c2, r2 = coords[j]
            if c1 == c2 or r1 == r2 or abs(c1 - c2) == abs(r1 - r2):
                attacks += 1
    return attacks

def print_board_text(coords, n, step, heuristics):
    """Draw the board with column and row numbers using 1-indexed headers."""
    board = [['-' for _ in range(n)] for _ in range(n)]
    for idx, (col, row) in enumerate(coords):
        if 0 <= col < n and 0 <= row < n:
            board[row][col] = f"Q{idx+1}"

    print(f"--- Step {step} | Mutually Attacking Pairs (Heuristics): {heuristics} ---")
    col_header = "    " + " ".join(f"{i:<3}" for i in range(1, n + 1))
    print(col_header)
    print("    " + "-" * (n * 4 - 1))
    for r_idx, row in enumerate(board):
        row_str = f"{r_idx + 1:<2}| " + " ".join(f"{item:<3}" for item in row)
        print(row_str)
    print()

def get_valid_one_step_moves(current_state, q_idx, n):
    """Generates all valid 1-step moves (up, down, left, right, diagonals) for a specific queen."""
    c_old, r_old = current_state[q_idx]
    moves = []
    directions = [
        (0, 1),   # Down
        (0, -1),  # Up
        (1, 0),   # Right
        (-1, 0),  # Left
        (1, 1),   # Diagonal Down-Right
        (1, -1),  # Diagonal Up-Right
        (-1, 1),  # Diagonal Down-Left
        (-1, -1)  # Diagonal Up-Left
    ]

    for dc, dr in directions:
        new_c = c_old + dc
        new_r = r_old + dr
        if 0 <= new_c < n and 0 <= new_r < n:
            if (new_c, new_r) not in current_state:
                neighbor_state = list(current_state)
                neighbor_state[q_idx] = (new_c, new_r)
                moves.append(neighbor_state)
    return moves

def describe_move_coords(old_state, new_state):
    for i in range(len(old_state)):
        if old_state[i] != new_state[i]:
            c_old, r_old = old_state[i]
            c_new, r_new = new_state[i]
            return f"Queen {i+1} moved from (Col {c_old+1}, Row {r_old+1}) to (Col {c_new+1}, Row {r_new+1})"
    return "No change"

def get_target_distance(state, target):
    """Computes distance to target configuration."""
    dist = 0
    for (c1, r1), (c2, r2) in zip(state, target):
        dist += abs(c1 - c2) + abs(r1 - r2)
    return dist

def hill_climbing_n_queens_coords():
    while True:
        try:
            n = int(input("Enter the value of N (board size, e.g., 4): ").strip())
            if n > 1:
                break
            print("Please enter an integer greater than 1.")
        except ValueError:
            print("Invalid input. Please enter a valid integer.")

    print(f"\nEnter custom start coordinates for {n} queens.")
    print(f"Use 1-indexed values from 1 to {n} in 'col,row' format:")
    current_state = []
    for i in range(n):
        while True:
            user_input = input(f"Enter position for Q{i+1} (e.g. 1,{i+1} or press Enter for default): ").strip()
            if not user_input:
                default_pos = (i, (i * 2) % n)
                current_state.append(default_pos)
                break
            else:
                try:
                    parts = user_input.split(',')
                    if len(parts) != 2:
                        raise ValueError("Must provide both column and row.")
                    col, row = map(lambda x: int(x) - 1, parts)
                    if 0 <= col < n and 0 <= row < n:
                        current_state.append((col, row))
                        break
                    else:
                        print(f"Coordinates must be within the board range 1 to {n}.")
                except ValueError:
                    print("Invalid input format. Please enter as 'col,row'.")

    current_h = get_heuristics_coords(current_state)

    step = 0
    print(f"\nInitial Custom Layout State (N={n}):")
    print_board_text(current_state, n, step, current_h)

    q_turn = 0
    no_improvement_count = 0
    max_steps = 1000

    # Target coordinates: q1=(2,1), q2=(4,2), q3=(1,3), q4=(3,4) -> 0-indexed: [(1, 0), (3, 1), (0, 2), (2, 3)]
    target_goal = [(1, 0), (3, 1), (0, 2), (2, 3)] if n == 4 else None

    while current_h > 0 and step < max_steps:
        q_label = f"Q{q_turn + 1}"
        one_step_states = get_valid_one_step_moves(current_state, q_turn, n)

        best_state = None
        best_metric = (float('inf'), float('inf'))

        for state in one_step_states:
            h_val = get_heuristics_coords(state)
            if target_goal:
                # If targeting a specific configuration, use distance to target to guide choices
                dist = get_target_distance(state, target_goal)
                metric = (dist, h_val)
            else:
                metric = (h_val, 0)

            if metric < best_metric:
                best_metric = metric
                best_state = state

        # Check if the move brings us closer to target or reduces attacks
        should_move = False
        if best_state is not None:
            if target_goal:
                curr_dist = get_target_distance(current_state, target_goal)
                best_dist = best_metric[0]
                if best_dist < curr_dist or best_metric[1] < current_h:
                    should_move = True
            elif best_metric[0] < current_h:
                should_move = True

        if should_move:
            move_description = describe_move_coords(current_state, best_state)
            current_state = best_state
            current_h = get_heuristics_coords(current_state)
            step += 1
            print(f"\nStep {step} (Turn: {q_label}): {move_description}")
            print_board_text(current_state, n, step, current_h)
            no_improvement_count = 0
        else:
            no_improvement_count += 1

        if no_improvement_count >= n:
            print(f"\n--- Local minimum reached. Initiating random restart --- ")
            all_squares = [(c, r) for c in range(n) for r in range(n)]
            current_state = random.sample(all_squares, n)
            current_h = get_heuristics_coords(current_state)
            step += 1
            print_board_text(current_state, n, f"{step} (Random Restart)", current_h)
            no_improvement_count = 0
            q_turn = 0
            continue

        q_turn = (q_turn + 1) % n

    if current_h == 0:
        one_indexed_coords = [(c + 1, r + 1) for c, r in current_state]
        print(f"\nSuccess! Solved state reached in {step} steps:")
        for i, (col, row) in enumerate(one_indexed_coords):
            print(f"q{i+1} = {col},{row}")
    else:
        print(f"\nStopped after reaching limit of {max_steps} steps without finding a solution.")

hill_climbing_n_queens_coords()


Enter the value of N (board size, e.g., 4): 4

Enter custom start coordinates for 4 queens.
Use 1-indexed values from 1 to 4 in 'col,row' format:
Enter position for Q1 (e.g. 1,1 or press Enter for default): 1,1
Enter position for Q2 (e.g. 1,2 or press Enter for default): 4,1
Enter position for Q3 (e.g. 1,3 or press Enter for default): 1,4
Enter position for Q4 (e.g. 1,4 or press Enter for default): 4,4

Initial Custom Layout State (N=4):
--- Step 0 | Mutually Attacking Pairs (Heuristics): 6 ---
    1   2   3   4  
    ---------------
1 | Q1  -   -   Q2 
2 | -   -   -   -  
3 | -   -   -   -  
4 | Q3  -   -   Q4 


Step 1 (Turn: Q1): Queen 1 moved from (Col 1, Row 1) to (Col 2, Row 1)
--- Step 1 | Mutually Attacking Pairs (Heuristics): 4 ---
    1   2   3   4  
    ---------------
1 | -   Q1  -   Q2 
2 | -   -   -   -  
3 | -   -   -   -  
4 | Q3  -   -   Q4 


Step 2 (Turn: Q2): Queen 2 moved from (Col 4, Row 1) to (Col 4, Row 2)
--- Step 2 | Mutually Attacking Pairs (Heuristics): 2 --